In [49]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped



polars.config.Config

In [50]:
market_response_path = "/Users/oceansxyz/alphanode-metal/logs/market/market_20260618_220038.jsonl"

#### Load JSONL - Binace Market WS

In [51]:
#Raw binance reponse for market web socket - pretty much a static
book_ticker_schema = pl.Struct([
    pl.Field("ts", pl.Int64),
    pl.Field("frame", pl.Struct([
        pl.Field("u", pl.Int64),
        pl.Field("s", pl.String),
        pl.Field("b", pl.String),
        pl.Field("B", pl.String),
        pl.Field("a", pl.String),
        pl.Field("A", pl.String),
    ])),
])

with open(market_response_path) as f:
    metadata_json = f.readline().strip()
    print("Metadata Raw Line:", metadata_json)
    df_market = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=book_ticker_schema))
        .unnest("json_str")
        .unnest("frame")
        .with_columns([
            pl.col("u").cast(pl.Int64),
            pl.col("s"),
            pl.col("b").cast(pl.Float64),
            pl.col("B").cast(pl.Float64),
            pl.col("a").cast(pl.Float64),
            pl.col("A").cast(pl.Float64),
        ])
    )


# Select relevant cols for alpha calulation with identifiable file names
market_rates = df_market.select(
    col("ts"), col("s").alias("symbol"), col("b").alias("bid"), col("a").alias("ask")
)

# duration of web socket listen
min_ts = market_rates.min().select("ts").item()
max_ts = market_rates.max().select("ts").item()
duration_m =(max_ts-min_ts)/60000


print(f"Duration of listen : {duration_m} mins")
print(market_rates.group_by("symbol").len())

Metadata Raw Line: {"ts":1781820039723,"frame":{"result":null,"id":1}}
Duration of listen : 28.413816666666666 mins
shape: (3, 2)
┌──────────┬───────┐
│ symbol   ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ BTCEURI  ┆ 938   │
│ BTCUSDT  ┆ 10466 │
│ EURIUSDT ┆ 190   │
└──────────┴───────┘


In [52]:
market_rates.head()

ts,symbol,bid,ask
i64,str,f64,f64
1781820039773,"""BTCUSDT""",62957.99,62958.0
1781820040495,"""BTCEURI""",54871.89,54891.46
1781820040615,"""BTCUSDT""",62957.99,62958.0
1781820040645,"""BTCUSDT""",62957.99,62958.0
1781820040767,"""BTCUSDT""",62957.99,62958.0


In [53]:
## Split dataframe symbolwise symbol is an orderbook
symbol_0 = market_rates.filter(col("symbol")=="BTCUSDT").select(["ts", col("bid").alias("BTCUSDT_bid"), col("ask").alias("BTCUSDT_ask")])
symbol_1 = market_rates.filter(col("symbol")=="BTCEURI").select(["ts", col("bid").alias("BTCEURI_bid"), col("ask").alias("BTCEURI_ask")])
symbol_2 = market_rates.filter(col("symbol")=="EURIUSDT").select(["ts", col("bid").alias("EURIUSDT_bid"), col("ask").alias("EURIUSDT_ask")])

In [54]:
# Forward fill missing values by copying the last known valid price down 
# for all generated ticker metrics simultaneously, skipping the timestamp.
market_rates_widened = symbol_0.join(symbol_1, on="ts", how="full", coalesce=True)\
    .join(symbol_2, on="ts", how="full", coalesce=True)\
    .sort("ts")\
    .select([
        col("ts"),
        col("*").exclude("ts").fill_null(strategy="forward")
    ]).drop_nulls()

market_rates_widened.head()

ts,BTCUSDT_bid,BTCUSDT_ask,BTCEURI_bid,BTCEURI_ask,EURIUSDT_bid,EURIUSDT_ask
i64,f64,f64,f64,f64,f64,f64
1781820048799,62957.99,62958.0,54871.89,54885.34,1.1471,1.1474
1781820048849,62957.99,62958.0,54871.89,54885.34,1.1471,1.1474
1781820049340,62957.99,62958.0,54871.89,54954.63,1.1471,1.1474
1781820049340,62957.99,62958.0,54871.89,54942.77,1.1471,1.1474
1781820049645,62957.99,62958.0,54871.89,54942.77,1.1471,1.1474


#### Notation & Model

NODE_0 = USDT NODE_1 = BTC NODE_2 = EURI  

EDGE_0 = BTCUSDT  EDGE_1 = BTCEURI  EDGE_2 = EURIUSDT  

Clockwise Walk  0→1  →1→2  →2→0  

BUY  BTCUSDT  @ASK  === 0→1  
SELL BTCEURI  @BID  === 1→2  
SELL EURIUSDT @BID  === 2→0


#### Clockwise Walk  0→1  →1→2  →2→0

In [55]:
clockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_ask"), "BTCEURI_bid", "EURIUSDT_bid")\
    .unique(subset=["BTCUSDT_ask", "BTCEURI_bid", "EURIUSDT_bid"])\
    .with_columns((col("BTCEURI_bid")*col("EURIUSDT_bid")).alias("BTCEURIUSDT_bid"))\
    .with_columns((col("BTCEURIUSDT_bid")/col("BTCUSDT_ask")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



clockwise_walk.sort("alpha", descending=True).head(20)



ts,BTCUSDT_ask,BTCEURI_bid,EURIUSDT_bid,BTCEURIUSDT_bid,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781821058598,62868.01,54831.18,1.1473,62907.812814,1.000633,6.331171
1781821166597,62797.87,54770.0,1.1473,62837.621,1.000633,6.329992
1781821180588,62779.2,54750.0,1.1473,62814.675,1.000565,5.650757
1781821183588,62779.21,54750.0,1.1473,62814.675,1.000565,5.649163
1781821183589,62779.22,54750.0,1.1473,62814.675,1.000565,5.647569
1781821183589,62779.26,54750.0,1.1473,62814.675,1.000564,5.641194
1781821183589,62779.46,54750.0,1.1473,62814.675,1.000561,5.609319
1781821183589,62779.74,54750.0,1.1473,62814.675,1.000556,5.564693
1781821057594,62872.86,54831.18,1.1473,62907.812814,1.000556,5.559285


#### AntiClockwise Walk   0←1 ←1←2  ←2←0

Remarkably profitable - SELL BTCUSDT; BUY BTCEURI; and BUY EURIUSDT

In [48]:
anticlockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_bid"), "BTCEURI_ask", "EURIUSDT_ask")\
    .unique(subset=["BTCUSDT_bid", "BTCEURI_ask", "EURIUSDT_ask"])\
    .with_columns((col("BTCEURI_ask")*col("EURIUSDT_ask")).alias("BTCEURIUSDT_ask"))\
    .with_columns((col("BTCUSDT_bid")/col("BTCEURIUSDT_ask")).alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



anticlockwise_walk.sort("alpha", descending=True).head(20)

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781820352585,62927.99,54853.1,1.1473,62932.96163,0.999921,-0.789988
1781820406295,62929.83,54850.0,1.1474,62934.89,0.99992,-0.804006
1781820409291,62929.83,54850.04,1.1474,62934.935896,0.999919,-0.811298
1781820411289,62929.83,54850.61,1.1474,62935.589914,0.999908,-0.915208
1781820412294,62929.83,54850.62,1.1474,62935.601388,0.999908,-0.917031
1781820409290,62929.83,54850.75,1.1474,62935.75055,0.999906,-0.940729
1781820353295,62927.99,54853.97,1.1473,62933.959781,0.999905,-0.948579
1781820405289,62929.83,54851.55,1.1474,62936.66847,0.999891,-1.086564
1781820399287,62929.83,54851.56,1.1474,62936.679944,0.999891,-1.088387


In [9]:
anticlockwise_walk.filter((col("ts")==1781820341584))

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64


In [47]:
anticlockwise_walk.filter((col("ts")<1781435059000) & (col("ts")>1781435057000))\
    .plot.line(x="ts", y="BTCUSDT_bid")

alt.Chart(...)